In [3]:
import pandas as pd
import numpy as np

Wczytanie bibliotek i danych

In [4]:
df = pd.read_csv("auction_results_color_svd.csv")

df.head()

,ARTIST,TECHNIQUE,SIGNATURE,CONDITION,TOTAL DIMENSIONS,YEAR,Colorfulness Score,SVD Entropy,PRICE
0,218,-1.295300,1,2,-0.157723,-1.039766,51.632554,5.453204,150
1,101,-0.122087,2,2,-0.442668,-0.580107,161.631656,6.154763,270
2,274,-0.122087,2,2,0.263423,-0.626073,117.464780,6.908661,360
3,354,3.397553,0,2,-0.827075,-0.488176,164.609302,6.986244,343
4,354,-0.122087,2,2,-0.145178,0.431142,91.023011,5.859255,150


Zmienne kategoryczne i podział na zbiory uczący i testowy

In [5]:
zmienne_kategoryczne =  ['ARTIST', 'TECHNIQUE', 'SIGNATURE', 'CONDITION']
bloki_kategoryczne = []
for zmienna in zmienne_kategoryczne:
    kody = df[zmienna].astype('category').cat.codes.to_numpy(dtype=np.int32)
    liczba_klas = np.max(kody) + 1
    one_hot = np.eye(liczba_klas)[kody]
    bloki_kategoryczne.append(one_hot)


X_kat_cale = np.hstack(bloki_kategoryczne)

In [6]:
np.random.seed(42)
df_train = df.sample(frac=0.8, random_state=42)
df_test = df.drop(df_train.index)

indeksy_train = df_train.index
indeksy_test = df_test.index

X_kat_train = X_kat_cale[indeksy_train]
X_kat_test = X_kat_cale[indeksy_test]

Normalizacja zmiennych

In [7]:
zmienne_liczbowe = ["TOTAL DIMENSIONS", "YEAR", "Colorfulness Score", "SVD Entropy"]
srednia = df_train[zmienne_liczbowe].mean()
odchylenie = df_train[zmienne_liczbowe].std()

df_train[zmienne_liczbowe] = (df_train[zmienne_liczbowe] - srednia) / odchylenie
df_test[zmienne_liczbowe] = (df_test[zmienne_liczbowe] - srednia) / odchylenie

Przejście na tablice NumPy

In [8]:
X_num_train = df_train[zmienne_liczbowe].to_numpy(dtype=np.float32)
X_num_test = df_test[zmienne_liczbowe].to_numpy(dtype=np.float32)

X_train = np.hstack([X_kat_train, X_num_train], dtype=np.float32)
X_test = np.hstack([X_kat_test, X_num_test], dtype=np.float32)

srednia_cena = df_train['PRICE'].mean()
odchylenie_cena = df_train['PRICE'].std()

y_train = ((df_train['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)
y_test = ((df_test['PRICE'] - srednia_cena) / odchylenie_cena).to_numpy(dtype=np.float32).reshape(-1, 1)

Funkcja do obliczania prawdziwych cen

In [9]:
def prawdziwa_cena(znormalizowana_cena):
    return znormalizowana_cena * odchylenie_cena + srednia_cena

Import bibliotek i metryk

In [16]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import time

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

Uczenie modeli i ewaluacja

In [17]:
import time
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Definiujemy modele
modele = {
    "Regresja Liniowa": LinearRegression(),
    "Regresja Ridge (L2)": Ridge(alpha=1.0),
    "Drzewo Decyzyjne": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Las Losowy": RandomForestRegressor(n_estimators=100, random_state=42),
    "K-Najbliższych Sąsiadów (KNN)": KNeighborsRegressor(n_neighbors=5)
}

# Spłaszczamy rzeczywiste ceny testowe
y_test_rzeczywiste = prawdziwa_cena(y_test.ravel())

print("--- KLASYCZNE MODELE ML ---\n")

for nazwa, model in modele.items():
    # Pomiar czasu start
    start_time = time.time()
    
    # Trening
    model.fit(X_train, y_train.ravel())
    
    # Pomiar czasu stop
    czas_treningu = time.time() - start_time
    
    # Predykcja i ewaluacja
    y_pred_znormalizowane = model.predict(X_test)
    y_pred_rzeczywiste = prawdziwa_cena(y_pred_znormalizowane)
    
    mae = mean_absolute_error(y_test_rzeczywiste, y_pred_rzeczywiste)
    rmse = np.sqrt(mean_squared_error(y_test_rzeczywiste, y_pred_rzeczywiste))
    r2 = r2_score(y_test_rzeczywiste, y_pred_rzeczywiste)
    
    print(f"Model: {nazwa}")
    print(f"  Czas treningu : {czas_treningu:.4f} s")
    print(f"  MAE           : {mae:.2f}")
    print(f"  RMSE          : {rmse:.2f}")
    print(f"  R^2           : {r2:.4f}")
    print("-" * 40)

--- KLASYCZNE MODELE ML ---

Model: Regresja Liniowa
  Czas treningu : 0.3012 s
  MAE           : 173.76
  RMSE          : 386.73
  R^2           : 0.3240
----------------------------------------
Model: Regresja Ridge (L2)
  Czas treningu : 0.0633 s
  MAE           : 172.20
  RMSE          : 386.73
  R^2           : 0.3240
----------------------------------------
Model: Drzewo Decyzyjne
  Czas treningu : 0.5499 s
  MAE           : 154.47
  RMSE          : 395.43
  R^2           : 0.2932
----------------------------------------
Model: Las Losowy
  Czas treningu : 97.0980 s
  MAE           : 99.86
  RMSE          : 258.17
  R^2           : 0.6987
----------------------------------------
Model: K-Najbliższych Sąsiadów (KNN)
  Czas treningu : 0.0040 s
  MAE           : 127.88
  RMSE          : 351.63
  R^2           : 0.4411
----------------------------------------


Sieć neuronowa Tensorflow

In [ ]:
import tensorflow as tf

print("--- SIEĆ NEURONOWA (TENSORFLOW/KERAS) ---\n")

# 1. Budowa architektury sieci
model_nn = tf.keras.Sequential([
    # Warstwa wejściowa + pierwsza ukryta (64 neurony, funkcja aktywacji ReLU)
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    
    # Druga warstwa ukryta (32 neurony)
    tf.keras.layers.Dense(32, activation='relu'),
    
    # Warstwa wyjściowa (1 neuron, liniowa aktywacja bo to regresja - przewidujemy jedną liczbę)
    tf.keras.layers.Dense(1)
])

# 2. Kompilacja modelu (wybór optymalizatora i funkcji straty - Mean Squared Error)
model_nn.compile(optimizer='adam', loss='mse')

# 3. Trening sieci z pomiarem czasu
start_time_nn = time.time()

# Trenujemy przez 50 epok, używając paczek (batch) po 32 próbki. 
# verbose=0 ukrywa paski postępu, żeby nie zaśmiecać notatnika
history = model_nn.fit(
    X_train, 
    y_train, 
    epochs=100, 
    batch_size=256, 
    verbose=0 
)

czas_treningu_nn = time.time() - start_time_nn

# 4. Predykcja i ewaluacja
# Keras zwraca predykcje w formacie 2D, więc używamy ravel() do spłaszczenia
y_pred_nn_znormalizowane = model_nn.predict(X_test, verbose=0).ravel()
y_pred_nn_rzeczywiste = prawdziwa_cena(y_pred_nn_znormalizowane)

mae_nn = mean_absolute_error(y_test_rzeczywiste, y_pred_nn_rzeczywiste)
rmse_nn = np.sqrt(mean_squared_error(y_test_rzeczywiste, y_pred_nn_rzeczywiste))
r2_nn = r2_score(y_test_rzeczywiste, y_pred_nn_rzeczywiste)

print(f"Model: Sieć Neuronowa (MLP)")
print(f"  Czas treningu : {czas_treningu_nn:.4f} s")
print(f"  MAE           : {mae_nn:.2f}")
print(f"  RMSE          : {rmse_nn:.2f}")
print(f"  R^2           : {r2_nn:.4f}")
print("-" * 40)

--- SIEĆ NEURONOWA (TENSORFLOW/KERAS) ---



c:\Users\Jan\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: Sieć Neuronowa (MLP)
  Czas treningu : 22.6987 s
  MAE           : 119.05
  RMSE          : 288.92
  R^2           : 0.6227
----------------------------------------
